# 3DGS Paper Landscape — Visualizations

Loads `3dgs_papers.db` (built by `build_vec_db.ipynb`) and produces:

1. **UMAP scatter** — full-abstract embeddings in 2D, clustered with HDBSCAN, interactive Plotly hover
2. **Per-section UMAP** — same layout, re-colored for each of the 8 rhetorical moves
3. **Similarity heatmap** — pairwise cosine distances across all papers
4. **Section divergence** — papers where *motivation* and *contribution* embeddings are furthest apart (novel framing) vs closest (incremental work)
5. **Paper recommender** — given a paper, surface nearest neighbors per section

In [ ]:
!pip install -q sqlite-vec umap-learn hdbscan plotly scikit-learn pandas numpy

In [ ]:
# ── Upload DB ─────────────────────────────────────────────────────────────────
try:
    from google.colab import files
    uploaded = files.upload()   # select 3dgs_papers.db
    DB_PATH = list(uploaded.keys())[0]
    print(f'Uploaded: {DB_PATH}')
except ImportError:
    DB_PATH = '3dgs_papers.db'
    print(f'Not in Colab — using: {DB_PATH}')

In [ ]:
# ── Load embeddings from DB ───────────────────────────────────────────────────
import sqlite3
import sqlite_vec
import struct
import numpy as np
import pandas as pd

con = sqlite3.connect(DB_PATH)
con.enable_load_extension(True)
sqlite_vec.load(con)
con.enable_load_extension(False)

SECTION_COLS = [
    'topic', 'motivation', 'contribution', 'detail_nuance',
    'evidence_contribution_2', 'weaker_result', 'narrow_impact', 'broad_impact'
]

# Load metadata
meta = pd.read_sql('SELECT * FROM papers ORDER BY id', con)
print(f'Papers: {len(meta)}')

# Infer embedding dimension from first row
raw_blob, = con.execute('SELECT embedding FROM vec_full WHERE paper_id = 1').fetchone()
DIM = len(raw_blob) // 4
print(f'Embedding dim: {DIM}')

def blob_to_vec(b: bytes) -> np.ndarray:
    return np.array(struct.unpack(f'{len(b)//4}f', b), dtype=np.float32)

# Full-abstract embeddings
rows = con.execute('SELECT paper_id, embedding FROM vec_full ORDER BY paper_id').fetchall()
paper_ids   = [r[0] for r in rows]
full_vecs   = np.stack([blob_to_vec(r[1]) for r in rows])   # (N, DIM)

# Per-section embeddings — dict: section -> (N, DIM)
section_vecs = {}
for sec in SECTION_COLS:
    rows_s = con.execute(
        'SELECT paper_id, embedding FROM vec_sections WHERE section = ? ORDER BY paper_id',
        [sec]
    ).fetchall()
    section_vecs[sec] = np.stack([blob_to_vec(r[1]) for r in rows_s])

# Align meta to the order of paper_ids
id_to_idx = {pid: i for i, pid in enumerate(paper_ids)}
meta = meta.set_index('id').loc[paper_ids].reset_index()
N = len(meta)
print(f'Loaded {N} papers with {DIM}-dim embeddings')

In [ ]:
# ── Fit UMAP on full-abstract embeddings ─────────────────────────────────────
import umap

print('Fitting UMAP...')
reducer = umap.UMAP(
    n_neighbors=12,
    min_dist=0.1,
    metric='cosine',
    random_state=42
)
coords_2d = reducer.fit_transform(full_vecs)   # (N, 2)
print(f'UMAP done: {coords_2d.shape}')

In [ ]:
# ── Cluster with HDBSCAN ──────────────────────────────────────────────────────
import hdbscan

clusterer = hdbscan.HDBSCAN(
    min_cluster_size=4,
    min_samples=2,
    metric='euclidean'   # on 2D UMAP coords
)
labels = clusterer.fit_predict(coords_2d)
n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
n_noise    = (labels == -1).sum()
print(f'Clusters: {n_clusters}  |  Noise points: {n_noise}')

meta['umap_x']   = coords_2d[:, 0]
meta['umap_y']   = coords_2d[:, 1]
meta['cluster']  = labels
meta['cluster_label'] = meta['cluster'].apply(
    lambda c: f'Cluster {c}' if c >= 0 else 'Noise'
)

In [ ]:
# ── 1. UMAP scatter — full abstract, colored by cluster ──────────────────────
import plotly.express as px

# Short label for hover: filename slug + first 80 chars of topic
meta['hover_title'] = meta['filename'].str.replace('-', ' ').str.title()
meta['hover_topic'] = meta['topic'].str[:120]

fig = px.scatter(
    meta,
    x='umap_x', y='umap_y',
    color='cluster_label',
    hover_name='hover_title',
    hover_data={'hover_topic': True, 'umap_x': False, 'umap_y': False, 'cluster_label': False},
    title='3DGS Paper Landscape — Full Abstract Embeddings (UMAP + HDBSCAN)',
    labels={'hover_topic': 'Topic', 'cluster_label': 'Cluster'},
    width=950, height=700,
    template='plotly_dark'
)
fig.update_traces(marker=dict(size=8, opacity=0.85))
fig.update_layout(legend_title_text='Cluster')
fig.show()

In [ ]:
# ── 2. Per-section UMAP — same 2D coords, re-colored by section-specific clusters
from sklearn.preprocessing import normalize
from plotly.subplots import make_subplots
import plotly.graph_objects as go

# Project each section's embeddings onto the existing UMAP space
# (transform, not re-fit — keeps layout consistent for comparison)
SECTION_LABELS = {
    'topic':                   'Topic',
    'motivation':              'Motivation',
    'contribution':            'Contribution',
    'detail_nuance':           'Detail / Nuance',
    'evidence_contribution_2': 'Evidence / Contribution 2',
    'weaker_result':           'Weaker Result',
    'narrow_impact':           'Narrow Impact',
    'broad_impact':            'Broad Impact',
}

fig = make_subplots(
    rows=2, cols=4,
    subplot_titles=list(SECTION_LABELS.values()),
    horizontal_spacing=0.04,
    vertical_spacing=0.1
)

for i, (sec, label) in enumerate(SECTION_LABELS.items()):
    row, col = divmod(i, 4)
    sec_2d = reducer.transform(section_vecs[sec])

    sec_labels = hdbscan.HDBSCAN(min_cluster_size=4, min_samples=2).fit_predict(sec_2d)

    fig.add_trace(
        go.Scatter(
            x=sec_2d[:, 0], y=sec_2d[:, 1],
            mode='markers',
            marker=dict(size=6, color=sec_labels, colorscale='Turbo', opacity=0.8),
            text=meta['hover_title'],
            hovertemplate='<b>%{text}</b><br>' + label + ': %{customdata}<extra></extra>',
            customdata=meta[sec].str[:100],
            showlegend=False
        ),
        row=row+1, col=col+1
    )

fig.update_layout(
    title='Per-Section UMAP — How Papers Cluster Across Rhetorical Moves',
    template='plotly_dark',
    height=700, width=1200
)
fig.update_xaxes(showticklabels=False)
fig.update_yaxes(showticklabels=False)
fig.show()

In [ ]:
# ── 3. Pairwise cosine similarity heatmap ────────────────────────────────────
from sklearn.metrics.pairwise import cosine_similarity

sim_matrix = cosine_similarity(full_vecs)   # (N, N)

# Sort by cluster so similar papers are adjacent
sort_idx = meta['cluster'].argsort().values
sim_sorted = sim_matrix[sort_idx][:, sort_idx]
labels_sorted = meta['hover_title'].iloc[sort_idx].tolist()

fig = px.imshow(
    sim_sorted,
    x=labels_sorted, y=labels_sorted,
    color_continuous_scale='RdBu',
    zmin=0, zmax=1,
    title='Pairwise Cosine Similarity — Full Abstract Embeddings',
    width=900, height=900,
    template='plotly_dark'
)
fig.update_traces(hovertemplate='%{y}<br>vs %{x}<br>similarity: %{z:.3f}<extra></extra>')
fig.update_xaxes(showticklabels=False)
fig.update_yaxes(showticklabels=False)
fig.show()

In [ ]:
# ── 4. Section divergence — motivation vs contribution ────────────────────────
# Papers where these two embeddings are far apart = novel framing
# Papers where they are close = incremental / well-trodden path

from sklearn.metrics.pairwise import paired_distances

mot = section_vecs['motivation']
con_vecs = section_vecs['contribution']

divergence = paired_distances(mot, con_vecs, metric='cosine')
meta['mot_con_divergence'] = divergence

fig = px.scatter(
    meta.sort_values('mot_con_divergence'),
    x='umap_x', y='umap_y',
    color='mot_con_divergence',
    color_continuous_scale='RdYlGn_r',
    hover_name='hover_title',
    hover_data={
        'motivation':          True,
        'contribution':        True,
        'mot_con_divergence':  ':.3f',
        'umap_x': False, 'umap_y': False
    },
    title='Motivation ↔ Contribution Divergence<br><sup>Green = tightly aligned (incremental) | Red = far apart (novel framing)</sup>',
    labels={'mot_con_divergence': 'Cosine Distance'},
    width=950, height=700,
    template='plotly_dark'
)
fig.update_traces(marker=dict(size=9, opacity=0.85))
fig.show()

print('Most novel framing (motivation ≠ contribution):')
print(meta.nlargest(5, 'mot_con_divergence')[['filename', 'mot_con_divergence', 'motivation', 'contribution']].to_string())
print('\nMost incremental (motivation ≈ contribution):')
print(meta.nsmallest(5, 'mot_con_divergence')[['filename', 'mot_con_divergence', 'motivation', 'contribution']].to_string())

In [ ]:
# ── 5. Paper recommender — nearest neighbors per section ─────────────────────
from sklearn.metrics.pairwise import cosine_similarity as cos_sim

def recommend(filename_fragment: str, k: int = 5):
    """
    Given a partial filename, show the k most similar papers per section.
    Useful for: given a paper you're reading, what else should you read?
    """
    matches = meta[meta['filename'].str.contains(filename_fragment, case=False)]
    if matches.empty:
        print(f'No paper found matching: {filename_fragment!r}')
        print('\nAvailable filenames (sample):')
        for f in meta['filename'].sample(min(10, len(meta))).tolist():
            print(f'  {f}')
        return
    idx = matches.index[0]
    print(f'Reference paper: {meta.loc[idx, "filename"]}')
    print(f'Topic: {meta.loc[idx, "topic"][:120]}\n')

    for sec in SECTION_COLS:
        vecs = section_vecs[sec]
        sims = cos_sim(vecs[idx:idx+1], vecs)[0]
        top_k = np.argsort(sims)[::-1][1:k+1]   # skip self
        print(f'── {SECTION_LABELS[sec]} ──')
        for rank, j in enumerate(top_k, 1):
            print(f'  {rank}. [{sims[j]:.3f}] {meta.iloc[j]["filename"]}')
            print(f'       {meta.iloc[j][sec][:100]}')
        print()

# Use the first paper in the dataset as a guaranteed working example
# Swap the fragment for any partial filename from your dataset
example = meta['filename'].iloc[0]
recommend(example)